# Day 34 — Personalization / Few-shot Adaptation
Primary unit: repetition. `k=2,3` repetitions mỗi class.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
FEATURE_NPZ=Path('/content/drive/MyDrive/MyoLab-AI-data/day31/features.npz')
OUTPUT_DIR=Path('/content/drive/MyDrive/MyoLab-AI-data/day34'); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
K_VALUES=[2,3]; SEEDS=list(range(3401,3421))


## Bước 1 — Load và audit NPZ


In [ ]:
data=np.load(FEATURE_NPZ,allow_pickle=False)
required={'X','y','subject_ids','repetition_ids','dataset_ids'}
missing=required-set(data.files)
assert not missing, missing
assert len(set(data['dataset_ids'].tolist()))==1, 'POOLED_DATASET_BLOCKED'
assert np.isfinite(data['X']).all(), 'NONFINITE_FEATURE_MATRIX'


## Bước 2 — Few-shot split không overlap


In [ ]:
import sys
sys.path.insert(0,str(Path.cwd()/'ai-core'/'personalization'))
from day34.fewshot import make_fewshot_split
subject=sorted(set(data['subject_ids'].tolist()))[0]
cal_idx,eval_idx=make_fewshot_split(data['subject_ids'],data['y'],data['repetition_ids'],subject,2,3401)
assert set(cal_idx).isdisjoint(set(eval_idx))


## Bước 3 — Run P0/P1 và xuất paired deltas
P0 phải load exact frozen Day 32 model; P1 chỉ fit adaptation layer trên calibration repetitions.


In [ ]:
expected_outputs=['few-shot-protocol.json','few-shot-results.csv','per-subject-improvement.csv','personalization-vs-global.csv']
print(expected_outputs)
